## Objetivo del cuaderno

Este cuaderno unifica los flujos de `rent` y `sale` a partir del CSV consolidado generado por `idealistaAPI_raw_to_preprocess.ipynb`.

El trigger es `OPERATION`, con valores permitidos `"rent"` o `"sale"`.

Flujo:
- seleccionar operacion
- cargar el CSV consolidado correspondiente
- hacer una lectura informativa del dataset
- seleccionar y renombrar columnas
- enriquecer con distancias minimas a POIs
- exportar el CSV limpio final


In [ ]:
import sys
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)

cwd = Path.cwd().resolve()
project_root = next((p for p in [cwd, *cwd.parents] if (p / "src").exists()), None)
if project_root is None:
    raise RuntimeError("No se encontro la raiz del proyecto (carpeta 'src').")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))


## Trigger de operacion

Cambia `OPERATION` a `"rent"` o `"sale"` antes de ejecutar el cuaderno.


In [ ]:
OPERATION = "rent"

operation_config = {
    "rent": {
        "input_csv": Path("../../data/raw/idealistaAPI/preprocess/total_rent_cantabria.csv"),
        "output_csv": Path("../../data/processed/idealistaAPI/rent_homes_clean.csv"),
        "label": "alquiler",
    },
    "sale": {
        "input_csv": Path("../../data/raw/idealistaAPI/preprocess/total_sales_cantabria.csv"),
        "output_csv": Path("../../data/processed/idealistaAPI/sale_homes_clean.csv"),
        "label": "venta",
    },
}

if OPERATION not in operation_config:
    raise ValueError(f"Operacion no soportada: {OPERATION}")

INPUT_CSV = operation_config[OPERATION]["input_csv"]
OUTPUT_CSV = operation_config[OPERATION]["output_csv"]
OPERATION_LABEL = operation_config[OPERATION]["label"]

print(f"Operacion seleccionada: {OPERATION} ({OPERATION_LABEL})")
print(f"INPUT_CSV: {INPUT_CSV}")
print(f"OUTPUT_CSV: {OUTPUT_CSV}")


#### Import del modulo geoespacial

Se importa en una celda separada para aislar errores de entorno o ruta.


In [ ]:
from src.geospatial_expansion import agregar_distancias_minimas_poi


## Carga de datos

Se lee directamente el consolidado de la operacion seleccionada generado en `data/raw/idealistaAPI/preprocess`.


In [ ]:
df = pd.read_csv(INPUT_CSV)

print(f"CSV cargado: {INPUT_CSV}")
print(f"Forma del dataset consolidado: {df.shape}")
df.head()


## Lectura informativa del dataset


In [ ]:
dataset_overview = pd.Series(
    {
        "operacion": OPERATION,
        "numero_registros": int(len(df)),
        "numero_columnas": int(len(df.columns)),
        "propertyCode_unicos": int(df["propertyCode"].nunique(dropna=True)) if "propertyCode" in df.columns else None,
        "propertyCode_duplicados": int(df["propertyCode"].duplicated().sum()) if "propertyCode" in df.columns else None,
        "municipios_unicos": int(df["municipality"].nunique(dropna=True)) if "municipality" in df.columns else None,
        "precio_min": float(pd.to_numeric(df["price"], errors="coerce").min()) if "price" in df.columns else None,
        "precio_mediana": float(pd.to_numeric(df["price"], errors="coerce").median()) if "price" in df.columns else None,
        "precio_max": float(pd.to_numeric(df["price"], errors="coerce").max()) if "price" in df.columns else None,
        "superficie_media_m2": float(pd.to_numeric(df["size"], errors="coerce").mean()) if "size" in df.columns else None,
    }
)

dataset_overview.to_frame("valor")


In [ ]:
numeric_columns = [col for col in ["price", "size", "rooms", "bathrooms"] if col in df.columns]
df[numeric_columns].apply(pd.to_numeric, errors="coerce").describe().T


In [ ]:
null_summary = (
    df.isna().mean().mul(100).sort_values(ascending=False).rename("pct_nulos").to_frame()
)

null_summary[null_summary["pct_nulos"] > 0].head(15)


In [ ]:
print(f"Columnas totales: {df.columns}")

columns = [
    "propertyCode",
    "price",
    "size",
    "rooms",
    "bathrooms",
    "detailedType.typology",
    "detailedType.subTypology",
    "province",
    "municipality",
    "district",
    "latitude",
    "longitude",
    "floor",
    "exterior",
    "hasLift",
    "parkingSpace.hasParkingSpace",
    "newDevelopment",
]

df_reduced = df[columns].copy()
df_reduced.head()


In [ ]:
columnas_espanol = {
    "propertyCode": "codigo_inmueble",
    "price": "precio",
    "size": "superficie_construida_m2",
    "rooms": "numero_dormitorios",
    "bathrooms": "numero_banos",
    "detailedType.typology": "tipologia",
    "detailedType.subTypology": "subtipologia",
    "operation": "tipo_operacion",
    "province": "provincia",
    "municipality": "municipio",
    "district": "distrito",
    "latitude": "latitud",
    "longitude": "longitud",
    "floor": "planta",
    "exterior": "es_exterior",
    "hasLift": "tiene_ascensor",
    "parkingSpace.hasParkingSpace": "tiene_garaje",
    "newDevelopment": "obra_nueva",
}

df_clean = df_reduced.rename(columns=columnas_espanol).copy()

print(f"Forma del dataset listo para enriquecer: {df_clean.shape}")
df_clean.head()


In [ ]:
df_clean["municipio"].value_counts().head(10)


In [ ]:
df_expandido = agregar_distancias_minimas_poi(df_clean, ["playa", "supermercado", "colegio"])

df_expandido.head()


In [ ]:
OUTPUT_DIR = OUTPUT_CSV.parent
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df_expandido.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")
print(f"CSV limpio exportado en: {OUTPUT_CSV.resolve()}")
